In [1]:
import numpy as np
import pandas as pd
import pytensor
import mat73
import os
import glob

%matplotlib inline
%config InlineBackend.figure_format='retina'


In [ ]:
pd.set_option('display.max_rows', 100)

## to create the datasets by looping through rat folders, use this:

In [ ]:
folder = "/scratch/work/bahriz1/Thesis/data/per_rat"

base_cols = [
    "Trial", "RatID", "Date", "StimOn", "Correct", "InstruResp",
    "ReactionTime", "PeakVel", "PeakVelTime"
]

veltime_cols = [f"VelTime_{i}" for i in range(1, 363)]
veltrace_cols = [f"VelTrace_{i}" for i in range(1, 363)]

final_cols = base_cols + veltime_cols + veltrace_cols + ["CR_Label"]

after_T1_list = []
after_T3_list = []

for csv_path in glob.glob(os.path.join(folder, "*_allT.csv")):

    tmp_df = pd.read_csv(csv_path)

    # force same columns + same order
    tmp_df = tmp_df.reindex(columns=final_cols)

    after_T1_tmp = tmp_df[
        (tmp_df["CR_Label"].shift(1) == 1) &
        (tmp_df["Trial"] == tmp_df["Trial"].shift(1) + 1)
    ]

    after_T3_tmp = tmp_df[
        (tmp_df["CR_Label"].shift(1) == 3) &
        (tmp_df["Trial"] == tmp_df["Trial"].shift(1) + 1)
    ]

    after_T1_list.append(after_T1_tmp)
    after_T3_list.append(after_T3_tmp)

In [ ]:
after_T1_all = pd.concat(after_T1_list, ignore_index=True)
after_T3_all = pd.concat(after_T3_list, ignore_index=True)

after_T1_all = after_T1_all.reindex(columns=final_cols)
after_T3_all = after_T3_all.reindex(columns=final_cols)

after_T1_all.to_parquet("after_T1.parquet")
after_T3_all.to_parquet("after_T3.parquet")

## To create the datasets using the full data, run this:

In [ ]:
df = pd.read_csv('../data/allT.csv', dtype = {"Date":str})

## to find the faulty data where reaction time < 0.7 and remove them:

In [ ]:
#bad_sessions = df.loc[df["ReactionTime"] < 0.7,["RatID", "Date"]].drop_duplicates()

In [ ]:
# bad_sessions = set(
#     zip(
#         df.loc[df["ReactionTime"] < 0.7, "RatID"],
#         df.loc[df["ReactionTime"] < 0.7, "Date"]
#     )
# )

# mask = [
#     (rat, date) not in bad_sessions
#     for rat, date in zip(df["RatID"], df["Date"])
# ]

# df_clean = df[mask]

In [1]:
#to validate that the bad rats have wrecked sessions and wrecked sessions only:


baddies = {141  :  [11123, 21123, 31123, 81123, 91123, 301023, 311023],
142  :  [11123, 21123, 31123, 81123, 91123, 311023],
144  :  [11123, 21123, 31123, 61123, 71123, 81123, 91123, 101123, 131123, 311023],
146  :  [11123, 21123, 31123, 61123, 71123, 81123, 91123, 101123, 141123, 151123, 311023],
147  :  [31123, 91123, 201123, 221123, 271123, 291123],
164  :  [10324, 40324, 50324, 80324, 110324, 120324, 220224, 230224, 260224, 270224, 280224, 290224],
166  :  [10324, 40324, 40424, 50324, 50424, 70324, 80324, 80424, 90424, 100424, 110324, 110424, 120324, 120424, 130324, 140324, 150324, 150424, 160424, 170424, 180324, 180424, 190324, 190424, 200324, 210324, 220324, 220424, 230424, 240424, 250324, 260324, 270324, 280324, 290224, 290324],
167  :  [10324, 40324, 50324, 70324, 80324, 110324],
169  :  [10324, 40324, 50324, 210224, 220224, 230224, 260224, 270224, 280224, 290224],
170  :  [10324, 40324, 40424, 50324, 50424, 70324, 80324, 80424, 90424, 100424, 110424, 120424, 150424, 160424, 170424, 180424, 220224, 230224, 260224, 270224, 270324, 280224, 280324, 290224, 290324],
230  :  [31224, 41224, 50125, 51224, 60125, 70125, 80125, 101224, 111224, 121224, 131224, 161224, 171224, 181224, 191224, 201224, 261224, 271224, 281224],
232  :  [31224, 41224, 50125, 51224, 60125, 70125, 80125, 90125, 101224, 111224, 121224, 131224, 140125, 161224, 171224, 181224, 191224, 201224, 261224, 271224, 281224],
233  :  [50125, 60125, 70125, 80125, 90125, 120125, 130125, 131224, 140125, 161224, 171224, 181224, 191224, 201224, 261224, 271224, 281224],
234  :  [50125, 60125, 70125, 80125, 90125, 131224, 140125, 161224, 171224, 181224, 191224, 201224, 261224, 271224, 281224],
235  :  [50125, 121224, 131224, 161224, 171224, 181224, 191224, 201224, 261224, 271224, 281224]}
baddies = {
    rat: [str(date).zfill(6) for date in dates]
    for rat, dates in baddies.items()
}

for ID in baddies.keys():
    print(df[df['RatID'] == ID]['Date'].unique() == baddies[ID])


#so removing the bad rat Ids are enough!

NameError: name 'df' is not defined

In [ ]:
badRats = df[df['ReactionTime']<0.7]['RatID'].unique()

df_clean = df[~df['RatID'].isin(badRats)]
df_clean =  df_clean.reset_index(drop = True)
df_clean["GlobalTrialID"] = np.arange(len(df_clean))
df_clean.insert(0, "GlobalTrialID", df_clean.pop("GlobalTrialID"))

In [8]:
df_clean.to_parquet("../data/all_clean.parquet")

## Read the clean data and create T1 and T3

In [2]:
df_clean = pd.read_parquet("../data/all_clean.parquet")

In [9]:
after_T3 = df_clean[
    (df_clean['CR_Label'].shift(1) == 3) &
    (df_clean['Trial'] == df_clean['Trial'].shift(1) + 1) &
    (df_clean['RatID'] == df_clean['RatID'].shift(1)) #not needed but better to be safe
]

In [10]:
after_T1 = df_clean[
    (df_clean['CR_Label'].shift(1) == 1) &
    (df_clean['Trial'] == df_clean['Trial'].shift(1) + 1) &
    (df_clean['RatID'] == df_clean['RatID'].shift(1))&
    (df_clean['Date'] == df_clean['Date'].shift(1))
]

In [11]:
after_T1.to_parquet("../data/after_T1.parquet")
after_T3.to_parquet("../data/after_T3.parquet")

### Import the data and modify it for HSSM model

In [ ]:
T1 = pd.read_parquet("../data/after_T1.parquet")
T3 = pd.read_parquet("../data/after_T3.parquet")

In [ ]:
T1

$$Correct = 0 → go stim, omission  \tab     → response = 1, rt = -999\\
Correct = 1 → go stim, hit        \tab    → response = 1, rt = actual RT\\
Correct = 2 → no-go stim, rejection  \tab → response = -1, rt = -999\\
Correct = 3 → no-go stim, false alarm \tab → response = -1, rt = actual RT\\$$

In [ ]:
tmp_df = T1[["RatID", "Trial", "Date", "Correct", "ReactionTime"]].copy()

tmp_df = tmp_df.rename(columns={
    "RatID": "subj_idx",
    "ReactionTime": "rt"
})

tmp_df["response"] = np.where(
    tmp_df["Correct"].isin([0, 1]),
    1,     # go stimulus
    -1     # no-go stimulus
)

tmp_df["rt"] = tmp_df["rt"].fillna(-999)

tmp_df["deadline"] = 1.75

tmp_df = tmp_df[["subj_idx", "rt", "response", "deadline", "Trial", "Date", "Correct"]]
T1model_df = tmp_df[["subj_idx", "rt", "response", "deadline"]].copy()

In [ ]:
T1model_df = T1model_df.copy()

T1model_df["rt"] = T1model_df["rt"].astype("float32")
T1model_df["deadline"] = T1model_df["deadline"].astype("float32")
T1model_df["response"] = T1model_df["response"].astype("int32")
T1model_df["subj_idx"] = T1model_df["subj_idx"].astype("int32")

In [ ]:
T1model_df

In [ ]:
tmp_df = T3[["RatID", "Trial", "Date", "Correct", "ReactionTime"]].copy()

tmp_df = tmp_df.rename(columns={
    "RatID": "subj_idx",
    "ReactionTime": "rt"
})

tmp_df["response"] = np.where(
    tmp_df["Correct"].isin([0, 1]),
    1,     # go stimulus
    -1     # no-go stimulus
)

tmp_df["rt"] = tmp_df["rt"].fillna(-999)

tmp_df["deadline"] = 1.75

tmp_df = tmp_df[["subj_idx", "rt", "response", "deadline", "Trial", "Date", "Correct"]]
T3model_df = tmp_df[["subj_idx", "rt", "response", "deadline"]].copy()

In [ ]:
T3model_df = T3model_df.copy()

T3model_df["rt"] = T3model_df["rt"].astype("float32")
T3model_df["deadline"] = T3model_df["deadline"].astype("float32")
T3model_df["response"] = T3model_df["response"].astype("int32")
T3model_df["subj_idx"] = T3model_df["subj_idx"].astype("int32")